In [ ]:
# === Cell 1: Install Dependencies ===
# Run this cell first in a fresh kernel.

import sys
print("Python executable used by this notebook:", sys.executable)

%pip install --upgrade pip setuptools wheel
%pip install --upgrade \
    "gradio==5.49.1" \
    "gradio-client==1.13.3" \
    "pydantic>=2.11,<2.12" \
    "pillow>=10,<11" \
    "numpy>=1.26,<2.1" \
    "pandas>=2.2,<2.4" \
    "requests>=2.32,<3"

%pip install --upgrade torch torchvision torchaudio
%pip install --upgrade "regex>=2024.11.6" "transformers>=4.46,<4.47" "timm>=1.0,<2"

print("\nDependencies installed.")
print("Restart the kernel once, then choose Run All.")


In [ ]:
# === Cell 2: Configure LM Studio and test connectivity ===

import os
import json
import requests

os.environ["OPENAI_BASE_URL"] = "http://localhost:1234/v1"
os.environ["OPENAI_API_KEY"] = "lmstudio-key"
os.environ["MODEL"] = "mistral-7b-instruct-v0.2"

print("Config loaded:")
print("  OPENAI_BASE_URL:", os.environ["OPENAI_BASE_URL"])
print("  OPENAI_API_KEY :", "(set)")
print("  MODEL          :", os.environ["MODEL"])

try:
    resp = requests.get(
        f"{os.environ['OPENAI_BASE_URL'].rstrip('/')}/models",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
        timeout=8,
    )
    resp.raise_for_status()
    models = resp.json()
    print("\nLM Studio is reachable.")
    print(json.dumps(models, indent=2)[:800], "...\n")
except Exception as e:
    print("\nCould not reach LM Studio yet.")
    print("  - Is LM Studio running?")
    print("  - Is the API server turned on?")
    print("  - Is a chat model loaded?")
    print("Error:", e)


In [ ]:
# === Cell 3: Imports, nutrition database, and helper functions ===

from typing import List, Dict, Any
from PIL import Image
import numpy as np
import pandas as pd

NUTRITION_DB = {
    "salmon": {"calories": 208, "protein_g": 20, "fat_g": 13, "carbs_g": 0, "fiber_g": 0},
    "asparagus": {"calories": 20, "protein_g": 2.2, "fat_g": 0.1, "carbs_g": 3.9, "fiber_g": 2.1},
    "tomato": {"calories": 18, "protein_g": 0.9, "fat_g": 0.2, "carbs_g": 3.9, "fiber_g": 1.2},
    "rice": {"calories": 130, "protein_g": 2.7, "fat_g": 0.3, "carbs_g": 28, "fiber_g": 0.4},
    "pasta": {"calories": 131, "protein_g": 5, "fat_g": 1.1, "carbs_g": 25, "fiber_g": 1.3},
    "avocado": {"calories": 160, "protein_g": 2, "fat_g": 15, "carbs_g": 9, "fiber_g": 7},
    "egg": {"calories": 155, "protein_g": 13, "fat_g": 11, "carbs_g": 1.1, "fiber_g": 0},
    "chicken_breast": {"calories": 165, "protein_g": 31, "fat_g": 3.6, "carbs_g": 0, "fiber_g": 0},
    "beef_steak": {"calories": 271, "protein_g": 25, "fat_g": 19, "carbs_g": 0, "fiber_g": 0},
    "broccoli": {"calories": 34, "protein_g": 2.8, "fat_g": 0.4, "carbs_g": 7, "fiber_g": 2.6},
    "potato": {"calories": 77, "protein_g": 2, "fat_g": 0.1, "carbs_g": 17, "fiber_g": 2.2},
    "spinach": {"calories": 23, "protein_g": 2.9, "fat_g": 0.4, "carbs_g": 3.6, "fiber_g": 2.2},
    "lemon": {"calories": 29, "protein_g": 1.1, "fat_g": 0.3, "carbs_g": 9.3, "fiber_g": 2.8},
    "olive_oil": {"calories": 884, "protein_g": 0, "fat_g": 100, "carbs_g": 0, "fiber_g": 0},
}

CANDIDATE_LABELS = [
    "salmon", "asparagus", "tomato", "rice", "pasta", "avocado", "egg",
    "chicken breast", "beef steak", "broccoli", "potato", "spinach", "lemon",
    "olive oil", "cheeseburger", "pizza", "pancakes", "sushi", "tofu"
]

def to_safe_key(label: str) -> str:
    label = label.lower().strip().replace(" ", "_")
    mapping = {
        "chicken_breast": "chicken_breast",
        "chicken": "chicken_breast",
        "beef_steak": "beef_steak",
        "steak": "beef_steak",
    }
    return mapping.get(label, label)

def scale_nutrition(per100g: Dict[str, float], grams: float) -> Dict[str, float]:
    factor = grams / 100.0
    return {k: round(v * factor, 2) for k, v in per100g.items()}


In [ ]:
# === Cell 4: Load CLIP and rank food labels ===

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

clip_model = None
clip_processor = None

def get_clip_components():
    global clip_model, clip_processor
    if clip_model is None or clip_processor is None:
        try:
            from transformers import CLIPProcessor, CLIPModel
        except ImportError as e:
            raise ImportError(
                "Transformers could not import CLIP. Re-run Cell 1 and then restart the kernel. "
                f"Original error: {e}"
            ) from e

        clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
        clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
    return clip_model, clip_processor

def clip_rank_labels(pil_image: Image.Image, candidate_texts: List[str], top_k: int = 3):
    model, processor = get_clip_components()
    inputs = processor(
        text=[f"a photo of {t}" for t in candidate_texts],
        images=pil_image,
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy().flatten()

    ranked = sorted(zip(candidate_texts, probs), key=lambda x: x[1], reverse=True)[:top_k]
    return [{"label": lab, "prob": float(p)} for lab, p in ranked]

print(f"CLIP will run on: {DEVICE}")


In [ ]:
# === Cell 5: Plain-Python pipeline functions ===

def lmstudio_chat(system_prompt: str, user_prompt: str) -> str:
    """
    Direct OpenAI-compatible HTTP call to LM Studio.
    We fold the system guidance into one user message for broad local-model compatibility.
    """
    url = f"{os.environ['OPENAI_BASE_URL'].rstrip('/')}/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": os.environ["MODEL"],
        "temperature": 0.3,
        "max_tokens": 700,
        "messages": [
            {
                "role": "user",
                "content": (
                    "### System instruction (follow strictly):\n"
                    f"{system_prompt}\n\n"
                    "### User request:\n"
                    f"{user_prompt}"
                ),
            }
        ],
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"]

def detect_foods(image: Image.Image, top_k: int = 3) -> List[Dict[str, Any]]:
    return clip_rank_labels(image, CANDIDATE_LABELS, top_k=top_k)

def estimate_nutrition(detections: List[Dict[str, Any]], grams_map: Dict[str, float]) -> Dict[str, Any]:
    items = []
    totals = {"calories": 0, "protein_g": 0, "fat_g": 0, "carbs_g": 0, "fiber_g": 0}

    for d in detections:
        raw_label = d["label"]
        key = to_safe_key(raw_label)
        grams = float(grams_map.get(raw_label, grams_map.get(key, 0)))
        if grams <= 0 or key not in NUTRITION_DB:
            continue

        scaled = scale_nutrition(NUTRITION_DB[key], grams)
        items.append({"ingredient": raw_label, "grams": grams, **scaled})
        for metric in totals:
            totals[metric] += scaled[metric]

    totals = {k: round(v, 2) for k, v in totals.items()}
    return {"items": items, "totals": totals}

def build_recipe(labels: List[str], nutrition: Dict[str, Any], user_prefs: str, servings: int, calories_goal: int) -> str:
    system = "You are a professional chef and nutrition coach. Be concise, friendly, and practical."
    user = f"""
Detected ingredients (top-3): {labels}
Estimated totals (approx, for the whole dish): {nutrition['totals']}
User preferences/restrictions: {user_prefs}
Target calories per serving: ~{calories_goal}
Servings: {servings}

Write:
1) A brief 2-sentence assessment of the plate and nutrition.
2) A personalized recipe (ingredients list in grams + step-by-step instructions).
3) One tip to adjust calories up or down while keeping protein reasonable.
"""
    return lmstudio_chat(system, user)

def analyze_food_image(
    image: Image.Image,
    user_prefs: str,
    servings: int = 2,
    calories_goal: int = 600,
    grams_overrides: Dict[str, float] | None = None,
) -> Dict[str, Any]:
    detections = detect_foods(image, top_k=3)
    labels = [d["label"] for d in detections]

    default_grams = {
        label: 150.0 if any(word in label for word in ["salmon", "steak", "chicken"]) else 80.0
        for label in labels
    }
    grams_map = default_grams | (grams_overrides or {})
    nutrition = estimate_nutrition(detections, grams_map)
    recipe_text = build_recipe(labels, nutrition, user_prefs, servings, calories_goal)

    return {"detections": detections, "nutrition": nutrition, "recipe_text": recipe_text}


In [ ]:
# === Cell 6: Gradio app ===

import json
import pandas as pd
import gradio as gr
from PIL import Image

print("Gradio version:", gr.__version__)

def nutrition_df(nutrition: Dict[str, Any]) -> pd.DataFrame:
    items = nutrition.get("items", [])
    totals = nutrition.get("totals", {})

    if not items:
        return pd.DataFrame([{"ingredient": "TOTAL", **totals}])

    df = pd.DataFrame(items)

    if "grams" in df.columns:
        total_grams = pd.to_numeric(df["grams"], errors="coerce").fillna(0).sum()
    else:
        total_grams = 0

    total_row = {
        "ingredient": "TOTAL",
        "grams": total_grams,
        **totals,
    }

    return pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)


def run_pipeline(image, servings, calories_goal, dietary_prefs, grams_overrides_json):
    if image is None:
        raise gr.Error("Please upload an image before clicking Analyze & Create Recipe.")

    pil = image.convert("RGB")

    overrides = None
    if grams_overrides_json and grams_overrides_json.strip():
        try:
            overrides = json.loads(grams_overrides_json)
            if not isinstance(overrides, dict):
                raise ValueError("Overrides must be a JSON object.")
        except Exception as error:
            raise gr.Error(f"Invalid portion-overrides JSON: {error}")

    try:
        result = analyze_food_image(
            pil,
            user_prefs=dietary_prefs or "",
            servings=int(servings),
            calories_goal=int(calories_goal),
            grams_overrides=overrides,
        )
    except Exception as error:
        raise gr.Error(f"Food analysis failed: {error}") from error

    labels_json = json.dumps(result["detections"], indent=2, default=str)
    df = nutrition_df(result["nutrition"])
    recipe = result["recipe_text"]

    return labels_json, df, recipe


# Close an older copy if this cell is rerun.
try:
    gr.close_all()
except Exception:
    pass


with gr.Blocks(title="Simple Food Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## Simple Food Analyzer\n"
        "Upload a food photo, estimate nutrition, and generate a personalized recipe."
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_in = gr.Image(
                type="pil",
                label="Upload a food photo",
                height=320,
            )

            servings = gr.Slider(
                minimum=1,
                maximum=6,
                value=2,
                step=1,
                label="Servings",
            )

            calories_goal = gr.Slider(
                minimum=200,
                maximum=1200,
                value=600,
                step=50,
                label="Target Calories per Serving",
            )

            dietary_prefs = gr.Textbox(
                label="Dietary preferences or restrictions",
                placeholder="For example: high protein, no dairy",
            )

            grams_overrides = gr.Textbox(
                label="Optional portion overrides as JSON",
                placeholder='{"salmon": 180, "asparagus": 90}',
            )

            run_btn = gr.Button(
                "Analyze & Create Recipe",
                variant="primary",
            )

        with gr.Column(scale=1):
            labels_out = gr.Code(
                label="Top-3 detected foods",
                language="json",
            )

            nutrition_out = gr.Dataframe(
                label="Estimated Nutrition (approximate)",
                wrap=True,
                interactive=False,
            )

            recipe_out = gr.Markdown(
                label="Personalized Recipe"
            )

    run_btn.click(
        fn=run_pipeline,
        inputs=[
            image_in,
            servings,
            calories_goal,
            dietary_prefs,
            grams_overrides,
        ],
        outputs=[
            labels_out,
            nutrition_out,
            recipe_out,
        ],
        api_name=False,
    )


demo.launch(
    share=True,
    inline=False,
    inbrowser=False,
    prevent_thread_lock=True,
    show_error=True,
    show_api=False,
)


In [ ]:
# === Cell 7: Optional local test without opening the UI ===

from pathlib import Path
from IPython.display import display

sample_path = Path("/Users/armenpischdotchian/Desktop/food01.png")
if sample_path.exists():
    img = Image.open(sample_path).convert("RGB")
    display(img)
    result = analyze_food_image(img, user_prefs="high protein, no dairy", servings=2, calories_goal=600)
    print("Detections:", result["detections"])
    display(nutrition_df(result["nutrition"]))
    print(result["recipe_text"])
else:
    print("Sample image not found at:", sample_path)
    print("Tip: update sample_path to any local food image and rerun this cell.")
